In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, sys, glob

found = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
RAW = os.path.dirname(found[0]) if found else None
BASE = "/kaggle/working"
PROCESSED = f"{BASE}/data/processed"
SRC = f"{BASE}/code/business_entity_resolution/src"
OUTPUT = f"{BASE}/output"

for folder in [PROCESSED, SRC, OUTPUT]:
    os.makedirs(folder, exist_ok=True)
sys.path.append(SRC)
print("RAW:", RAW, "| BASE:", BASE)

In [ ]:
!pip install -q polars duckdb rapidfuzz lightgbm scikit-learn

In [ ]:
import polars as pl

files = ["train_source1", "train_source2", "train_source3", "train_ground_truth",
          "test_source1", "test_source2", "test_source3"]
for name in files:
    out_path = f"{PROCESSED}/{name}.parquet"
    if not os.path.exists(out_path):
        pl.scan_csv(f"{RAW}/{name}.tsv", separator="\t").sink_parquet(out_path)
for name in files:
    print(name, pl.scan_parquet(f"{PROCESSED}/{name}.parquet").select(pl.len()).collect().item())

In [ ]:
%%writefile /kaggle/working/code/business_entity_resolution/src/normalization.py
import polars as pl
import re
import unicodedata

LEGAL_SUFFIX_MAP = {
    r"\bpvt\b": "private", r"\bltd\b": "limited", r"\bcorp\b": "corporation",
    r"\bco\b": "company", r"\bllp\b": "llp", r"\bllc\b": "llc", r"\binc\b": "inc",
    r"\bsarl\b": "sarl", r"\bsas\b": "sas", r"\bsci\b": "sci",
    r"\beurl\b": "eurl", r"\bsa\b": "sa",
}
LEGAL_SUFFIX_TOKENS = {"private","limited","corporation","company","llp","llc","inc","ltd","pvt","corp","co","sarl","sas","sci","eurl","sa"}
INDIAN_STATES = {"tn":"tamil nadu","up":"uttar pradesh","rj":"rajasthan","dl":"delhi","ka":"karnataka","mh":"maharashtra","wb":"west bengal","gj":"gujarat","mp":"madhya pradesh","ap":"andhra pradesh","tg":"telangana","kl":"kerala","pb":"punjab","hr":"haryana","br":"bihar"}
STOPWORDS = LEGAL_SUFFIX_TOKENS | {"the", "and", "shri", "m/s", "ms"}

def strip_accents(text):
    if not text: return text
    return "".join(c for c in unicodedata.normalize("NFKD", text) if unicodedata.category(c) != "Mn")

def clean_legal_dots(text):
    return re.sub(r"\b([A-Za-z])\.(?=[A-Za-z]\.)", r"\1", text)

def strip_leading_junk(text):
    text = re.sub(r"^\[[^\]]*\]\s*", "", text)
    return re.sub(r"^[^\w\s]+\s*", "", text)

def dedupe_repeated_words(text):
    return re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", text, flags=re.IGNORECASE)

def remove_null_literals(text):
    if not text: return text
    return re.sub(r"\b(null|none|nan|undefined)\b", "", text, flags=re.IGNORECASE)

def expand_indian_state(text):
    if not text: return text
    return " ".join(INDIAN_STATES.get(w.lower(), w) for w in text.split())

def _apply_all_name_text_fixes(text):
    if not text: return ""
    text = strip_accents(text)
    text = clean_legal_dots(text)
    text = strip_leading_junk(text)
    text = dedupe_repeated_words(text)
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    for pattern, replacement in LEGAL_SUFFIX_MAP.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

def full_normalize_name(df: pl.DataFrame, col: str) -> pl.DataFrame:
    df = df.with_columns(pl.col(col).map_elements(_apply_all_name_text_fixes, return_dtype=pl.Utf8).alias(f"{col}_normalized"))
    df = df.with_columns(
        pl.col(f"{col}_normalized").str.split(" ")
          .list.eval(pl.element().filter(~pl.element().is_in(STOPWORDS)))
          .list.join(" ").alias(f"{col}_core")
    )
    df = df.with_columns(pl.col(f"{col}_core").str.replace_all(r"\s", "").alias(f"{col}_nospace"))
    return df

def _apply_all_address_fixes(text):
    if not text: return ""
    text = strip_accents(text)
    text = remove_null_literals(text)
    text = re.sub(r"\b(ph|phone|mob|tel)[:.]?\s*[\d,\s]+", " ", text, flags=re.IGNORECASE)
    text = text.lower().strip()
    text = expand_indian_state(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def full_normalize_address(df: pl.DataFrame, col: str = "business_address") -> pl.DataFrame:
    df = df.with_columns(pl.col(col).is_null().alias("is_address_missing"))
    df = df.with_columns(
        pl.col(col).fill_null("").map_elements(_apply_all_address_fixes, return_dtype=pl.Utf8).alias(f"{col}_cleaned")
    )
    return df

def extract_digits(text):
    if not text: return ""
    digits = re.findall(r"\d+", text)
    return " ".join(str(int(d)) for d in digits if d)

In [ ]:
%%writefile /kaggle/working/code/business_entity_resolution/src/scoring.py
import polars as pl

def explode_ground_truth(gt: pl.DataFrame, prefix: str) -> pl.DataFrame:
    return (
        gt.filter(pl.col("matched_entity_ids") != "")
        .select(["source1_entity_id", pl.col("matched_entity_ids").str.split(",")])
        .explode("matched_entity_ids")
        .with_columns(pl.col("matched_entity_ids").str.strip_chars())
        .rename({"source1_entity_id": "s1_id", "matched_entity_ids": "other_id"})
        .filter(pl.col("other_id").str.starts_with(prefix))
    )

def f_beta_score(precision, recall, beta=0.5):
    b2 = beta ** 2
    denom = (b2 * precision) + recall
    return (1 + b2) * (precision * recall) / denom if denom > 0 else 0.0

def compute_f05_macro(predictions: dict, ground_truth: dict) -> float:
    scores = []
    for s1_id, true_set in ground_truth.items():
        pred_set = predictions.get(s1_id, set())
        if len(true_set) == 0:
            scores.append(1.0 if len(pred_set) == 0 else 0.0)
            continue
        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        scores.append(f_beta_score(precision, recall, beta=0.5))
    return sum(scores) / len(scores) if scores else 0.0

In [ ]:
%%writefile /kaggle/working/code/business_entity_resolution/src/sampling.py
import polars as pl

def build_random_sample(source1_path, source2_path, source3_path, gt_path, n=20_000, seed=42):
    gt = pl.read_parquet(gt_path)
    source1 = pl.read_parquet(source1_path)
    sample_s1 = source1.sample(n=n, seed=seed)
    sample_gt = gt.filter(pl.col("source1_entity_id").is_in(sample_s1["entity_id"]))
    matched_ids = (
        sample_gt.filter(pl.col("matched_entity_ids") != "")
        .select(pl.col("matched_entity_ids").str.split(","))
        .explode("matched_entity_ids")["matched_entity_ids"].str.strip_chars().to_list()
    )
    source2 = pl.read_parquet(source2_path)
    source3 = pl.read_parquet(source3_path)
    sample_s2 = pl.concat([
        source2.filter(pl.col("entity_id").is_in(matched_ids)),
        source2.sample(n=min(50_000, source2.shape[0]), seed=seed)
    ]).unique(subset=["entity_id"])
    sample_s3 = pl.concat([
        source3.filter(pl.col("entity_id").is_in(matched_ids)),
        source3.sample(n=min(50_000, source3.shape[0]), seed=seed)
    ]).unique(subset=["entity_id"])
    return sample_s1, sample_s2, sample_s3, sample_gt

In [ ]:
EXCLUDE_WORDS = ['m','sri','mr','smt','dr','mrs','shri','ms','new','services','india',
                  'mumbai','global','group','international','management','enterprises',
                  'trading','company','industries','solutions','systems','associates',
                  'consultants','ventures','maison','ets','etablissements']
exclude_list_sql = "[" + ",".join(f"'{w}'" for w in EXCLUDE_WORDS) + "]"
print("Exclude list ready.")

In [ ]:
from sampling import build_random_sample
from normalization import full_normalize_name, full_normalize_address, extract_digits

s1_rand, s2_rand, s3_rand, gt_rand = build_random_sample(
    f"{PROCESSED}/train_source1.parquet", f"{PROCESSED}/train_source2.parquet",
    f"{PROCESSED}/train_source3.parquet", f"{PROCESSED}/train_ground_truth.parquet", n=20_000
)
for name, df in [("s1_rand", s1_rand), ("s2_rand", s2_rand), ("s3_rand", s3_rand)]:
    df = full_normalize_name(df, "business_name")
    df = full_normalize_address(df, "business_address")
    df = df.with_columns(pl.col("business_address_cleaned").map_elements(extract_digits, return_dtype=pl.Utf8).alias("address_digits"))
    if name == "s1_rand": s1_rand = df
    elif name == "s2_rand": s2_rand = df
    else: s3_rand = df
print("Sizes:", s1_rand.shape, s2_rand.shape, s3_rand.shape, gt_rand.shape)

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("PRAGMA memory_limit='10GB'")
con.execute("PRAGMA threads=4")

con.register("rand_s1_raw", s1_rand.to_pandas())
con.register("rand_s2_raw", s2_rand.to_pandas())
con.register("rand_s3_raw", s3_rand.to_pandas())

for src, dst in [("rand_s1_raw","rand_s1_keyed"), ("rand_s2_raw","rand_s2_keyed"), ("rand_s3_raw","rand_s3_keyed")]:
    con.execute(f"""
        CREATE OR REPLACE TABLE {dst} AS
        SELECT *,
               COALESCE(
                   list_extract(list_filter(string_split(business_name_core,' '), x -> NOT list_contains({exclude_list_sql}, x) AND length(x)>=2), 1),
                   list_extract(string_split(business_name_core,' '), 1)
               ) AS block_key,
               substr(address_digits, 1, 4) AS addr_key
        FROM {src}
    """)
    print(f"{dst} ready.")

In [ ]:
import time

def build_sql_features_fast(s1_table, other_table, con):
    return con.execute(f"""
        SELECT
            s1.entity_id AS s1_id, o.entity_id AS other_id,
            jaro_winkler_similarity(s1.business_name_normalized, o.business_name_normalized)*100 AS name_jw,
            (1 - levenshtein(s1.business_name_normalized, o.business_name_normalized)::FLOAT /
                 GREATEST(length(s1.business_name_normalized), length(o.business_name_normalized), 1))*100 AS name_lev_ratio,
            CASE WHEN s1.business_name_core = o.business_name_core THEN 1 ELSE 0 END AS name_core_exact,
            jaro_winkler_similarity(s1.business_address_cleaned, o.business_address_cleaned)*100 AS addr_jw,
            len(list_intersect(string_split(s1.address_digits,' '), string_split(o.address_digits,' '))) AS addr_digit_overlap,
            CAST(s1.is_address_missing OR o.is_address_missing AS INTEGER) AS addr_missing_either
        FROM {s1_table} s1 JOIN {other_table} o
          ON s1.country = o.country
         AND s1.block_key = o.block_key AND s1.block_key IS NOT NULL AND s1.block_key != ''
         AND ((s1.addr_key = o.addr_key AND s1.addr_key != '' AND o.addr_key != '') OR s1.addr_key='' OR o.addr_key='')
    """).df()

feat_s2 = build_sql_features_fast("rand_s1_keyed", "rand_s2_keyed", con)
feat_s3 = build_sql_features_fast("rand_s1_keyed", "rand_s3_keyed", con)
print("Feature shapes:", feat_s2.shape, feat_s3.shape)

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
import joblib
import scoring
importlib_dummy = None  # (no reload needed, fresh session)

SQL_FEATURE_COLS = ["name_jw","name_lev_ratio","name_core_exact","addr_jw","addr_digit_overlap","addr_missing_either"]

feat_s2_pl = pl.from_pandas(feat_s2)
feat_s3_pl = pl.from_pandas(feat_s3)
train_s2 = feat_s2_pl.join(scoring.explode_ground_truth(gt_rand,"S2-").with_columns(pl.lit(1).alias("label")), on=["s1_id","other_id"], how="left").with_columns(pl.col("label").fill_null(0))
train_s3 = feat_s3_pl.join(scoring.explode_ground_truth(gt_rand,"S3-").with_columns(pl.lit(1).alias("label")), on=["s1_id","other_id"], how="left").with_columns(pl.col("label").fill_null(0))
train_all = pl.concat([train_s2, train_s3])
print("Label balance:\n", train_all["label"].value_counts())

X = train_all.select(SQL_FEATURE_COLS).to_pandas()
y = train_all["label"].to_pandas()
groups = train_all["s1_id"].to_pandas()

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(X))
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    m = LGBMClassifier(n_estimators=300, is_unbalance=True, random_state=42)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof[val_idx] = m.predict_proba(X.iloc[val_idx])[:, 1]
    print(f"Fold {fold} done.")

train_all = train_all.with_columns(pl.Series("match_probability", oof))
final_model = LGBMClassifier(n_estimators=300, is_unbalance=True, random_state=42)
final_model.fit(X, y)
joblib.dump(final_model, f"{OUTPUT}/final_model.joblib")
print("Model trained and saved.")

In [ ]:
def build_predictions(df, threshold):
    filtered = df.filter(pl.col("match_probability") >= threshold)
    preds = {}
    for s1_id, group in filtered.group_by("s1_id"):
        preds[s1_id[0]] = set(group["other_id"].to_list())
    return preds

ground_truth_dict = {}
for row in gt_rand.iter_rows(named=True):
    raw = row["matched_entity_ids"]
    ground_truth_dict[row["source1_entity_id"]] = set(x.strip() for x in raw.split(",") if x.strip()) if raw else set()

best_t, best_s = 0.5, 0.0
for t in np.arange(0.3, 0.99, 0.05):
    score = scoring.compute_f05_macro(build_predictions(train_all, t), ground_truth_dict)
    print(f"Threshold {t:.2f} -> F_0.5: {score:.4f}")
    if score > best_s:
        best_s, best_t = score, t

BEST_THRESHOLD = best_t
FINAL_MODEL = final_model
FEATURE_COLS_FINAL = SQL_FEATURE_COLS
print(f"\nLOCKED IN — Threshold: {BEST_THRESHOLD} | F_0.5: {best_s:.4f}")

In [ ]:
TEST_NORM_DIR = f"{PROCESSED}/test_normalized"
os.makedirs(TEST_NORM_DIR, exist_ok=True)

def normalize_and_save(name):
    out_path = f"{TEST_NORM_DIR}/{name}.parquet"
    if os.path.exists(out_path):
        return pl.read_parquet(out_path)
    df = pl.read_parquet(f"{PROCESSED}/{name}.parquet")
    df = full_normalize_name(df, "business_name")
    df = full_normalize_address(df, "business_address")
    df = df.with_columns(pl.col("business_address_cleaned").map_elements(extract_digits, return_dtype=pl.Utf8).alias("address_digits"))
    df.write_parquet(out_path)
    return df

test_s1 = normalize_and_save("test_source1")
test_s2 = normalize_and_save("test_source2")
test_s3 = normalize_and_save("test_source3")
print("Test shapes:", test_s1.shape, test_s2.shape, test_s3.shape)

In [ ]:
con.register("test_s1", test_s1.to_pandas())

for label, df_var in [("s2", test_s2), ("s3", test_s3)]:
    con.register(f"test_{label}_raw", df_var.to_pandas())
    con.execute(f"""
        CREATE OR REPLACE TABLE {label}_table AS
        SELECT *,
               COALESCE(
                   list_extract(list_filter(string_split(business_name_core,' '), x -> NOT list_contains({exclude_list_sql}, x) AND length(x)>=2), 1),
                   list_extract(string_split(business_name_core,' '), 1)
               ) AS block_key,
               substr(address_digits, 1, 4) AS addr_key
        FROM test_{label}_raw
    """)
    print(f"{label}_table ready.")

con.execute("""
    CREATE OR REPLACE TABLE test_s1 AS
    SELECT *,
           COALESCE(
               list_extract(list_filter(string_split(business_name_core,' '), x -> NOT list_contains(%s, x) AND length(x)>=2), 1),
               list_extract(string_split(business_name_core,' '), 1)
           ) AS block_key,
           substr(address_digits, 1, 4) AS addr_key
    FROM test_s1
""" % exclude_list_sql)
print("test_s1 keyed and ready.")

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE test_s1_keyed AS
    SELECT *,
           COALESCE(
               list_extract(list_filter(string_split(business_name_core,' '), x -> NOT list_contains(%s, x) AND length(x)>=2), 1),
               list_extract(string_split(business_name_core,' '), 1)
           ) AS block_key,
           substr(address_digits, 1, 4) AS addr_key
    FROM test_s1
""" % exclude_list_sql)
print("test_s1_keyed ready.")

# Sanity check — confirm the columns actually exist this time
print(con.execute("DESCRIBE test_s1_keyed").df()[["column_name"]])

In [ ]:
BLOCK_CHUNK_DIR = f"{OUTPUT}/blocking_chunks_final"
os.makedirs(BLOCK_CHUNK_DIR, exist_ok=True)
CHUNK_SIZE = 20_000
n_chunks = (test_s1.shape[0] // CHUNK_SIZE) + 1
print(f"{n_chunks} chunks total.")

def block_combined(s1_chunk_ids, table_name, con):
    ids_df = pl.DataFrame({"entity_id": s1_chunk_ids})
    con.register("id_filter", ids_df.to_pandas())
    result = con.execute(f"""
        SELECT DISTINCT s1.entity_id AS s1_id, t.entity_id AS other_id
        FROM test_s1_keyed s1
        JOIN id_filter f ON s1.entity_id = f.entity_id
        JOIN {table_name} t
          ON s1.country = t.country AND s1.block_key = t.block_key
         AND s1.block_key IS NOT NULL AND s1.block_key != ''
         AND ((s1.addr_key = t.addr_key AND s1.addr_key!='' AND t.addr_key!='') OR s1.addr_key='' OR t.addr_key='')
    """).df()
    con.unregister("id_filter")
    return pl.from_pandas(result)

test_s1_ids = pl.read_parquet(f"{TEST_NORM_DIR}/test_source1.parquet")["entity_id"].to_list()

for i in range(n_chunks):
    out_s2 = f"{BLOCK_CHUNK_DIR}/s2_chunk_{i}.parquet"


    
    out_s3 = f"{BLOCK_CHUNK_DIR}/s3_chunk_{i}.parquet"
    if os.path.exists(out_s2) and os.path.exists(out_s3):
        continue
    chunk_ids = test_s1_ids[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE]
    start = time.time()
    block_combined(chunk_ids, "s2_table", con).write_parquet(out_s2)
    block_combined(chunk_ids, "s3_table", con).write_parquet(out_s3)
    if i % 10 == 0:
        print(f"Chunk {i+1}/{n_chunks} done in {time.time()-start:.1f}s")
print("BLOCKING COMPLETE.")

In [ ]:
SCORED_DIR = f"{OUTPUT}/scored_chunks"
os.makedirs(SCORED_DIR, exist_ok=True)

def score_chunk(chunk_path, other_table, con, model, feature_cols):
    chunk = pl.read_parquet(chunk_path)
    if chunk.shape[0] == 0:
        return pl.DataFrame({"s1_id":[], "other_id":[], "match_probability":[]})
    con.register("real_cand", chunk.to_pandas())
    result = con.execute(f"""
        SELECT c.s1_id, c.other_id,
            jaro_winkler_similarity(s1.business_name_normalized, o.business_name_normalized)*100 AS name_jw,
            (1 - levenshtein(s1.business_name_normalized, o.business_name_normalized)::FLOAT /
                 GREATEST(length(s1.business_name_normalized), length(o.business_name_normalized),1))*100 AS name_lev_ratio,
            CASE WHEN s1.business_name_core = o.business_name_core THEN 1 ELSE 0 END AS name_core_exact,
            jaro_winkler_similarity(s1.business_address_cleaned, o.business_address_cleaned)*100 AS addr_jw,
            len(list_intersect(string_split(s1.address_digits,' '), string_split(o.address_digits,' '))) AS addr_digit_overlap,
            CAST(s1.is_address_missing OR o.is_address_missing AS INTEGER) AS addr_missing_either
        FROM real_cand c
        JOIN test_s1 s1 ON c.s1_id = s1.entity_id
        JOIN {other_table} o ON c.other_id = o.entity_id
    """).df()
    con.unregister("real_cand")
    feat = pl.from_pandas(result)
    X = feat.select(feature_cols).to_pandas()
    probs = model.predict_proba(X)[:, 1]
    return feat.select(["s1_id","other_id"]).with_columns(pl.Series("match_probability", probs.round(4)))

overall_start = time.time()
for i in range(n_chunks):
    out_path = f"{SCORED_DIR}/scored_chunk_{i}.parquet"
    s2_path = f"{BLOCK_CHUNK_DIR}/s2_chunk_{i}.parquet"
    s3_path = f"{BLOCK_CHUNK_DIR}/s3_chunk_{i}.parquet"
    if os.path.exists(out_path):
        for p in [s2_path, s3_path]:
            if os.path.exists(p): os.remove(p)
        continue

    scored_s2 = score_chunk(s2_path, "s2_table", con, FINAL_MODEL, FEATURE_COLS_FINAL).with_columns(pl.lit("S2").alias("source"))
    scored_s3 = score_chunk(s3_path, "s3_table", con, FINAL_MODEL, FEATURE_COLS_FINAL).with_columns(pl.lit("S3").alias("source"))
    combined = pl.concat([scored_s2, scored_s3])
    combined.write_parquet(out_path, compression="zstd")

    if os.path.exists(s2_path): os.remove(s2_path)
    if os.path.exists(s3_path): os.remove(s3_path)

    if i % 10 == 0 or i == n_chunks - 1:
        print(f"Chunk {i+1}/{n_chunks} | total: {(time.time()-overall_start)/60:.1f}min | rows: {combined.shape[0]}")

print(f"SCORING COMPLETE in {(time.time()-overall_start)/60:.1f} min.")

In [ ]:
import glob as glob_mod

all_scored = pl.concat([pl.read_parquet(f) for f in glob_mod.glob(f"{SCORED_DIR}/*.parquet")])
print("Total scored candidate rows:", all_scored.shape[0])

# candidate_pairs.tsv — ALL blocked+scored candidates per s1 entity
candidate_pairs = (
    all_scored.group_by("s1_id")
    .agg(pl.col("other_id").unique().str.concat(",").alias("candidate_entity_ids"))
)

# matching_results.tsv — only above-threshold, per s1 entity
matches = all_scored.filter(pl.col("match_probability") >= BEST_THRESHOLD)
matching_results = (
    matches.group_by("s1_id")
    .agg(pl.col("other_id").unique().str.concat(",").alias("matched_entity_ids"))
)

# Ensure EVERY test s1 entity appears, including singletons (empty string)
all_s1_ids = pl.DataFrame({"s1_id": test_s1_ids})
matching_results_full = all_s1_ids.join(matching_results, on="s1_id", how="left").with_columns(
    pl.col("matched_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})
candidate_pairs_full = all_s1_ids.join(candidate_pairs, on="s1_id", how="left").with_columns(
    pl.col("candidate_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})

matching_results_full.write_csv(f"{OUTPUT}/matching_results.tsv", separator="\t")
candidate_pairs_full.write_csv(f"{OUTPUT}/candidate_pairs.tsv", separator="\t")
print("Files written:", matching_results_full.shape, candidate_pairs_full.shape)

In [1]:
import os
print("test_s1_ids defined:", 'test_s1_ids' in dir())
print("BEST_THRESHOLD defined:", 'BEST_THRESHOLD' in dir())

test_s1_ids defined: False
BEST_THRESHOLD defined: False


In [2]:
import os, sys, glob
found = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
RAW = os.path.dirname(found[0]) if found else None
BASE = "/kaggle/working"
PROCESSED = f"{BASE}/data/processed"
SRC = f"{BASE}/code/business_entity_resolution/src"
OUTPUT = f"{BASE}/output"
sys.path.append(SRC)

In [3]:
import polars as pl
import joblib

MODEL_PATH = f"{OUTPUT}/final_model.joblib"
FINAL_MODEL = joblib.load(MODEL_PATH)
FEATURE_COLS_FINAL = ["name_jw","name_lev_ratio","name_core_exact","addr_jw","addr_digit_overlap","addr_missing_either"]

In [4]:
TEST_NORM_DIR = f"{PROCESSED}/test_normalized"
test_s1 = pl.read_parquet(f"{TEST_NORM_DIR}/test_source1.parquet")
test_s1_ids = test_s1["entity_id"].to_list()
print("test_s1 loaded:", test_s1.shape)

test_s1 loaded: (1732544, 10)


In [5]:
SCORED_DIR = f"{OUTPUT}/scored_chunks"
print("Scored chunks on disk:", len(os.listdir(SCORED_DIR)) if os.path.exists(SCORED_DIR) else 0)

Scored chunks on disk: 87


In [ ]:
import polars as pl

# Lazy scan across ALL scored chunk files — does NOT load everything into memory at once
scored_lazy = pl.scan_parquet(f"{SCORED_DIR}/*.parquet")

# ---- candidate_pairs.tsv: aggregate ALL candidates per s1 entity ----
candidate_pairs = (
    scored_lazy
    .group_by("s1_id")
    .agg(pl.col("other_id").unique().str.concat(",").alias("candidate_entity_ids"))
    .collect(engine="streaming")
)
print("candidate_pairs aggregated:", candidate_pairs.shape)

# ---- matching_results.tsv: filter to threshold FIRST (lazily), then aggregate ----
matching_results = (
    scored_lazy
    .filter(pl.col("match_probability") >= BEST_THRESHOLD)
    .group_by("s1_id")
    .agg(pl.col("other_id").unique().str.concat(",").alias("matched_entity_ids"))
    .collect(engine="streaming")
)
print("matching_results aggregated:", matching_results.shape)

# ---- Ensure every test S1 entity appears, including singletons ----
all_s1_ids_df = pl.DataFrame({"s1_id": test_s1_ids})

matching_results_full = all_s1_ids_df.join(matching_results, on="s1_id", how="left").with_columns(
    pl.col("matched_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})

candidate_pairs_full = all_s1_ids_df.join(candidate_pairs, on="s1_id", how="left").with_columns(
    pl.col("candidate_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})

matching_results_full.write_csv(f"{OUTPUT}/matching_results.tsv", separator="\t")
candidate_pairs_full.write_csv(f"{OUTPUT}/candidate_pairs.tsv", separator="\t")
print("Files written:", matching_results_full.shape, candidate_pairs_full.shape)

/tmp/ipykernel_1699/2335894876.py:10: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .agg(pl.col("other_id").unique().str.concat(",").alias("candidate_entity_ids"))


In [1]:
import os, sys, glob
found = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
RAW = os.path.dirname(found[0]) if found else None
BASE = "/kaggle/working"
PROCESSED = f"{BASE}/data/processed"
OUTPUT = f"{BASE}/output"
SCORED_DIR = f"{OUTPUT}/scored_chunks"

import polars as pl
TEST_NORM_DIR = f"{PROCESSED}/test_normalized"
test_s1 = pl.read_parquet(f"{TEST_NORM_DIR}/test_source1.parquet")
test_s1_ids = test_s1["entity_id"].to_list()

BEST_THRESHOLD = 0.85  # confirm this matches your actual locked-in value

print("test_s1 loaded:", test_s1.shape)
print("Scored chunks:", len(os.listdir(SCORED_DIR)))

test_s1 loaded: (1732544, 10)
Scored chunks: 87


In [2]:
import duckdb

con2 = duckdb.connect()
con2.execute("PRAGMA memory_limit='8GB'")
con2.execute("PRAGMA threads=2")

print("Aggregating candidate_pairs (all candidates per entity)...")
candidate_pairs = con2.execute(f"""
    SELECT s1_id, string_agg(DISTINCT other_id, ',') AS candidate_entity_ids
    FROM read_parquet('{SCORED_DIR}/*.parquet')
    GROUP BY s1_id
""").df()
print("candidate_pairs aggregated:", candidate_pairs.shape)

print("Aggregating matching_results (threshold-filtered)...")
matching_results = con2.execute(f"""
    SELECT s1_id, string_agg(DISTINCT other_id, ',') AS matched_entity_ids
    FROM read_parquet('{SCORED_DIR}/*.parquet')
    WHERE match_probability >= {BEST_THRESHOLD}
    GROUP BY s1_id
""").df()
print("matching_results aggregated:", matching_results.shape)

# Small, safe joins in Polars — ensures every test entity appears, singletons included
candidate_pairs_pl = pl.from_pandas(candidate_pairs)
matching_results_pl = pl.from_pandas(matching_results)
all_s1_ids_df = pl.DataFrame({"s1_id": test_s1_ids})

matching_results_full = all_s1_ids_df.join(matching_results_pl, on="s1_id", how="left").with_columns(
    pl.col("matched_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})

candidate_pairs_full = all_s1_ids_df.join(candidate_pairs_pl, on="s1_id", how="left").with_columns(
    pl.col("candidate_entity_ids").fill_null("")
).rename({"s1_id": "source1_entity_id"})

matching_results_full.write_csv(f"{OUTPUT}/matching_results.tsv", separator="\t")
candidate_pairs_full.write_csv(f"{OUTPUT}/candidate_pairs.tsv", separator="\t")
print("Files written:", matching_results_full.shape, candidate_pairs_full.shape)

Aggregating candidate_pairs (all candidates per entity)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (9.8 GiB/9.8 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [3]:
import os, glob
import polars as pl
import duckdb

BASE = "/kaggle/working"
PROCESSED = f"{BASE}/data/processed"
OUTPUT = f"{BASE}/output"
SCORED_DIR = f"{OUTPUT}/scored_chunks"
TEST_NORM_DIR = f"{PROCESSED}/test_normalized"

test_s1 = pl.read_parquet(f"{TEST_NORM_DIR}/test_source1.parquet")
test_s1_ids = test_s1["entity_id"].to_list()
BEST_THRESHOLD = 0.85

print("Total test entities:", len(test_s1_ids))

Total test entities: 1732544


In [4]:
import time

BATCH_SIZE = 100_000   # process 100K S1 entities at a time
n_batches = (len(test_s1_ids) // BATCH_SIZE) + 1

matching_out_path = f"{OUTPUT}/matching_results.tsv"
candidate_out_path = f"{OUTPUT}/candidate_pairs.tsv"

# Write headers once, then append each batch — never hold the full output in memory
with open(matching_out_path, "w") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
with open(candidate_out_path, "w") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")

overall_start = time.time()

for b in range(n_batches):
    batch_ids = test_s1_ids[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    if not batch_ids:
        continue

    con_b = duckdb.connect()  # fresh, lightweight connection per batch — no leftover state
    con_b.execute("PRAGMA memory_limit='6GB'")
    con_b.execute("PRAGMA threads=2")

    ids_df = pl.DataFrame({"s1_id": batch_ids})
    con_b.register("batch_ids", ids_df.to_pandas())

    # Only scan rows relevant to THIS batch, via the WHERE IN join — much smaller working set
    cand_result = con_b.execute(f"""
        SELECT s.s1_id, string_agg(DISTINCT s.other_id, ',') AS candidate_entity_ids
        FROM read_parquet('{SCORED_DIR}/*.parquet') s
        JOIN batch_ids b ON s.s1_id = b.s1_id
        GROUP BY s.s1_id
    """).df()

    match_result = con_b.execute(f"""
        SELECT s.s1_id, string_agg(DISTINCT s.other_id, ',') AS matched_entity_ids
        FROM read_parquet('{SCORED_DIR}/*.parquet') s
        JOIN batch_ids b ON s.s1_id = b.s1_id
        WHERE s.match_probability >= {BEST_THRESHOLD}
        GROUP BY s.s1_id
    """).df()

    con_b.close()

    cand_pl = pl.from_pandas(cand_result)
    match_pl = pl.from_pandas(match_result)
    batch_ids_df = pl.DataFrame({"s1_id": batch_ids})

    match_full = batch_ids_df.join(match_pl, on="s1_id", how="left").with_columns(
        pl.col("matched_entity_ids").fill_null("")
    ).rename({"s1_id": "source1_entity_id"})
    cand_full = batch_ids_df.join(cand_pl, on="s1_id", how="left").with_columns(
        pl.col("candidate_entity_ids").fill_null("")
    ).rename({"s1_id": "source1_entity_id"})

    # Append (no header) to the output files
    with open(matching_out_path, "a") as f:
        for row in match_full.iter_rows():
            f.write(f"{row[0]}\t{row[1]}\n")
    with open(candidate_out_path, "a") as f:
        for row in cand_full.iter_rows():
            f.write(f"{row[0]}\t{row[1]}\n")

    print(f"Batch {b+1}/{n_batches} done ({len(batch_ids)} entities) | total elapsed: {(time.time()-overall_start)/60:.1f}min")

print(f"\nDONE. Total time: {(time.time()-overall_start)/60:.1f} min")
print("Files:", matching_out_path, candidate_out_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Batch 1/18 done (100000 entities) | total elapsed: 2.1min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Batch 17/18 done (100000 entities) | total elapsed: 32.3min


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OSError: [Errno 28] No space left on device